In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
import sqlite3
import pandas as pd

from src.ai_classifier import classify_unmatched_rows

In [6]:
conn = sqlite3.connect("../data/bdc_tracker.db")
unmatched = pd.read_sql("SELECT * FROM unmatched_positions", conn)
conn.close()

print(unmatched.shape)
print(unmatched["period_end_date"].value_counts().sort_index())

(456, 7)
period_end_date
2024-06-30    411
2024-09-30      2
2024-12-31      3
2025-03-31      3
2025-06-30     11
2025-09-30      8
2025-12-31      9
2026-03-31      9
Name: count, dtype: int64


In [4]:
classified_full = classify_unmatched_rows(unmatched, sample_size=None)
print(classified_full.shape)
print(classified_full["investment_type"].value_counts())

Classifying rows 0-14 of 456...
  Batch cost: $0.0567 (running total: $0.0567)
Classifying rows 15-29 of 456...
  Batch cost: $0.0598 (running total: $0.1165)
Classifying rows 30-44 of 456...
  Batch cost: $0.0567 (running total: $0.1732)
Classifying rows 45-59 of 456...
  Batch cost: $0.0620 (running total: $0.2352)
Classifying rows 60-74 of 456...
  Batch cost: $0.0623 (running total: $0.2975)
Classifying rows 75-89 of 456...
  Batch cost: $0.0619 (running total: $0.3594)
Classifying rows 90-104 of 456...
  Batch cost: $0.0625 (running total: $0.4219)
Classifying rows 105-119 of 456...
  Batch cost: $0.0624 (running total: $0.4843)
Classifying rows 120-134 of 456...
  Batch cost: $0.0622 (running total: $0.5465)
Classifying rows 135-149 of 456...
  Batch cost: $0.0627 (running total: $0.6092)
Classifying rows 150-164 of 456...
  Batch cost: $0.0627 (running total: $0.6719)
Classifying rows 165-179 of 456...
  Batch cost: $0.0619 (running total: $0.7338)
Classifying rows 180-194 of 45

In [7]:
import sqlite3

conn = sqlite3.connect("../data/bdc_tracker.db")
positions = pd.read_sql("SELECT * FROM positions", conn)
conn.close()

mm_final = positions[
    positions["investment_name"].astype(str).str.contains("BlackRock|Fidelity|State Street", case=False, na=False)
]
print(mm_final[["investment_name", "fair_value", "period_end_date"]].to_string())

                                                                    investment_name fair_value period_end_date
693                                                  BlackRock ICS US Treasury Fund        166      2026-03-31
694                  Fidelity Investments Money Market Treasury Portfolio - Class I          2      2026-03-31
695   State Street Institutional U.S. Government Money Market Fund - Investor Class      7,158      2026-03-31
696    State Street Institutional U.S. Government Money Market Fund - Premier Class     38,156      2026-03-31
1687  State Street Institutional U.S. Government Money Market Fund - Investor Class      6,807      2025-12-31
1688   State Street Institutional U.S. Government Money Market Fund - Premier Class     17,168      2025-12-31
1689                                                 BlackRock ICS US Treasury Fund      3,167      2025-12-31
2686  State Street Institutional U.S. Government Money Market Fund - Investor Class        440      2025-09-30
2

In [5]:
classified_full.to_csv("../data/processed/bxsl_ai_classified.csv", index=False)

conn = sqlite3.connect("../data/bdc_tracker.db")
classified_full.to_sql("ai_classified_positions", conn, if_exists="replace", index=False)
conn.close()

print("저장 완료")

저장 완료
